In [56]:
import pandas as pd
import gspread

from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pymongo import MongoClient
from oauth2client.service_account import ServiceAccountCredentials # 특정 서비스에 접근할 수 있게 해준다 (여기서는 구글에 접근하기 위함)


SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1nVwlhO0tuE-vj2IOJzF7dCTmcFGWAZ_RT9E5-6DskUM/edit?usp=sharing"
GOOGLE_JSON_FILE = "google_service_account.json"

# mysql 연결
db_user = "root"
db_password = "ksdir8558"
db_host = "localhost"
db_name = "wconcept_db_260423"

engine = create_engine(
    f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}?charset=utf8mb4"
)

# mongodb 연결
mongo_client = MongoClient("mongodb://localhost:27017/")
mongo_db = mongo_client["wconcept_db_260423"]
blog_collection = mongo_db["blog_posts"]

# brand_summary -1
brand_sql = """
SELECT
    B.brand_id,
    B.brand_name,
    COUNT(DISTINCT P.product_id) AS product_count,
    ROUND(AVG(M.sale_price), 0) AS avg_sale_price,
    ROUND(AVG(M.rating), 2) AS avg_rating,
    SUM(M.review_count) AS total_review_count,
    SUM(M.like_count) AS total_like_count
FROM brands B
LEFT JOIN products P
    ON B.brand_id = P.brand_id
LEFT JOIN product_metrics M
    ON P.product_id = M.product_id
GROUP BY B.brand_id, B.brand_name
ORDER BY B.brand_id
"""

mysql_brand_df = pd.read_sql(brand_sql, engine)

pipline = [
    {
        "$group": {
            "_id": "$brand_id",
            "blog_post_count": {"$sum": 1},
            "unique_blogger_count": {"$addToSet": "$bloggername"},
            "latest_post_date": {"$max": "$postdate"}
        }
    }
]

mongo_result = list(blog_collection.aggregate(pipline))
mongo_df = pd.DataFrame(mongo_result)

if not mongo_df.empty :
    mongo_df = mongo_df.rename(columns={"_id": "brand_id"})
    mongo_df["unique_blogger_count"] = mongo_df["unique_blogger_count"].apply(len)
else :
    mongo_df = pd.DataFrame(columns=[
        "brand_id", "blog_post_count", "unique_blogger_count", "latest_post_date"
    ])

brand_summary_df = pd.merge(
    mysql_brand_df, 
    mongo_df,
    on="brand_id",
    how="left"
)

numeric_cols = [
    "product_count",
    "avg_sale_price",
    "avg_rating",
    "total_review_count",
    "total_like_count",
    "blog_post_count",
    "unique_blogger_count"
]

for col in numeric_cols :
    if col in brand_summary_df.columns :
        brand_summary_df[col] = brand_summary_df[col].fillna(0)
if "latest_post_date" in brand_summary_df.columns :
    brand_summary_df["latest_post_date"] = brand_summary_df["latest_post_date"].fillna("")

brand_summary_df= brand_summary_df.fillna("")

int_cols = [
    "avg_sale_price",
    "total_review_count",
    "total_like_count",
    "blog_post_count",
    "unique_blogger_count"
]

for col in int_cols :
    if col in brand_summary_df.columns :
        brand_summary_df[col] = brand_summary_df[col].astype(int)

# blog_posts_datail -2
docs = list(
   blog_collection.find(
    {}, # 조건이 없음 / 필터화하지 않음
    {
        "_id": 0,
        "brand_id": 1,
        "brand_name" : 1,
        "query": 1,
        "source": 1,
        "title": 1,
        "link": 1,
        "description": 1,
        "bloggername": 1,
        "bloggerlink": 1,
        "postdate": 1,
        "collected_at": 1
    }
    ) 
)

blog_datail_df = pd.DataFrame(docs)

if not blog_datail_df.empty :
    if "postdate" in blog_datail_df.columns :
        blog_datail_df["postdate"] = pd.to_datetime(blog_datail_df["postdate"], format="%Y%y%d", errors="coerce").dt.strftime("%Y-%m-%d")
    if "collected_at" in blog_datail_df.columns :
        blog_datail_df["collected_at"] = pd.to_datetime(blog_datail_df["collected_at"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

    blog_datail_df = blog_datail_df.fillna("")
    blog_datail_df = blog_datail_df.sort_values(
        ["brand_id", "postdate"],
        ascending=[True, False]
    )
else :
    blog_datail_df = pd.DataFrame(columns=[
        "brand_id", "brand_name", "query", "source", "title", "link", "description", "bloggername", "bloggerlink", "postdate", "collected_at"
    ])



# product_summary -3
product_sql = """
SELECT
    B.brand_id,
    B.brand_name,
    P.product_id,
    P.product_no,
    P.product_url,
    MAX(M.sale_price) AS sale_price,
    MAX(M.original_price) AS original_price,
    MAX(M.discount_rate) AS discount_rate,
    MAX(M.rating) AS rating,
    MAX(M.review_count) AS review_count,
    MAX(M.like_count) AS like_count,
    MAX(M.crawl_at) AS latest_crawl_at
FROM brands B
LEFT JOIN products P 
    ON B.brand_id = P.brand_id
LEFT JOIN product_metrics M
    ON P.product_id = M.product_id
GROUP BY 
    B.brand_id,
    B.brand_name,
    P.product_id,
    P.product_no,
    p.product_name,
    P.product_url
ORDER BY B.brand_name, P.product_name
"""

product_summary_df = pd.read_sql(product_sql, engine)

if "latest_crawl_at" in product_summary_df.columns :
    product_summary_df["latest_crawl_at"] = pd.to_datetime(product_summary_df["latest_crawl_at"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
    product_summary_df["latest_crawl_at"] = product_summary_df["latest_crawl_at"].fillna("")
    
scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

creds = ServiceAccountCredentials.from_json_keyfile_name(
    GOOGLE_JSON_FILE, 
    scope
)

client = gspread.authorize(creds)
spreadsheet = client.open_by_url(SPREADSHEET_URL)

def get_or_create_worksheet(spreadsheet, title, rows=1000, cols=20) :
    try :
        ws = spreadsheet.worksheet(title)
    except : 
        ws = spreadsheet.add_worksheet(title=title, rows=rows, cols=cols)
    return ws

def upload_dataframe_to_sheet(spreadsheet, sheet_name, df, rows=1000, cols=20) :
    ws = get_or_create_worksheet(spreadsheet, sheet_name, rows=rows, cols=cols)
    ws.clear()

    df = df.fillna("")
    data = [df.columns.tolist()] + df.values.tolist()

    ws.update(values=data, range_name="A1")
    print(f"{sheet_name} 탭 업로드 완료")

upload_dataframe_to_sheet(
    spreadsheet,
    "brand_summary",
    brand_summary_df,
    rows=max(len(brand_summary_df) + 10, 1000),
    cols=max(len(brand_summary_df.columns) + 5, 20)
)
upload_dataframe_to_sheet(
    spreadsheet,
    "blog_datail_df",
    blog_datail_df,
    rows=max(len(blog_datail_df) + 10, 1000),
    cols=max(len(blog_datail_df.columns) + 5, 20)
)
upload_dataframe_to_sheet(
    spreadsheet,
    "product_summary",
    product_summary_df,
    rows=max(len(product_summary_df) + 10, 1000),
    cols=max(len(product_summary_df.columns) + 5, 20)
)

print("모든 탭 업로드 완료")

brand_summary 탭 업로드 완료
blog_datail_df 탭 업로드 완료
product_summary 탭 업로드 완료
모든 탭 업로드 완료


In [46]:
product_summary_df

,brand_id,brand_name,product_id,product_no,product_name
0,45,그레이스유,125,305735254,[단 하루!] Joy Cardigan (9 Color)
1,45,그레이스유,186,305735263,[단 하루!] Joy Sleeveless Top (9 Color)
2,45,그레이스유,45,305921882,[단 하루!] Kaley One-piece (Black)
3,45,그레이스유,181,302830140,[단 하루!] Zozo Cardigan (5 Color)
4,128,그로브,128,308502917,26SS CALIA SKIRT PANTS (BEIGE)
...,...,...,...,...,...
192,122,헤이,122,306641564,[한정특가] [도경수 협찬][W컨셉 단독]pebble long snake chain...
193,67,호카,67,308181985,HOKA 여성 샌들 호카 호파라 2 블랙 1147670-BBLC
194,161,혼스비,161,307972675,BLUE MOUNTAINS [BLACK]
195,21,휠라,21,306356882,[차정원 PICK] 에샤페 실버문_1XM02348H_063


In [52]:
blog_datail_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 141 entries, 4 to 136
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   brand_id      141 non-null    int64         
 1   brand_name    141 non-null    object        
 2   query         141 non-null    object        
 3   source        141 non-null    object        
 4   title         141 non-null    object        
 5   link          141 non-null    object        
 6   description   141 non-null    object        
 7   bloggername   141 non-null    object        
 8   bloggerlink   141 non-null    object        
 9   postdate      141 non-null    object        
 10  collected_at  141 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(9)
memory usage: 13.2+ KB


In [13]:
mysql_brand_df

,brand_id,brand_name,product_count,avg_sale_price,avg_rating,total_review_count,total_like_count
0,3,아디다스,12,79797.0,4.97,9320.0,95236.0
1,5,우포스,3,45697.0,4.97,2388.0,3276.0
2,6,비에이유 바이 브라이드앤유,10,266543.0,2.47,1456.0,66078.0
3,7,쿠쿠,3,229169.0,3.33,26.0,982.0
4,8,모노로우,2,128440.0,4.95,1018.0,33904.0
...,...,...,...,...,...,...,...
101,194,꼼파뇨,1,29744.0,5.00,140.0,4026.0
102,196,웰노운,1,87296.0,0.00,0.0,32.0
103,197,허비쉬,1,102520.0,0.00,0.0,26.0
104,198,로스,1,167200.0,0.00,0.0,228.0


In [21]:
brand_summary_df.shape

(106, 10)

In [22]:
brand_summary_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   brand_id              106 non-null    int64  
 1   brand_name            106 non-null    object 
 2   product_count         106 non-null    int64  
 3   avg_sale_price        106 non-null    int64  
 4   avg_rating            106 non-null    float64
 5   total_review_count    106 non-null    int64  
 6   total_like_count      106 non-null    int64  
 7   blog_post_count       106 non-null    int64  
 8   unique_blogger_count  106 non-null    int64  
 9   latest_post_date      106 non-null    object 
dtypes: float64(1), int64(7), object(2)
memory usage: 8.4+ KB


In [25]:
blog_datail_df.shape

(141, 11)

In [31]:
blog_datail_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141 entries, 0 to 140
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   brand_id      141 non-null    int64         
 1   brand_name    141 non-null    object        
 2   query         141 non-null    object        
 3   source        141 non-null    object        
 4   title         141 non-null    object        
 5   link          141 non-null    object        
 6   description   141 non-null    object        
 7   bloggername   141 non-null    object        
 8   bloggerlink   141 non-null    object        
 9   postdate      141 non-null    datetime64[ns]
 10  collected_at  141 non-null    datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(8)
memory usage: 12.2+ KB


In [33]:
blog_datail_df

,brand_id,brand_name,query,source,title,link,description,bloggername,bloggerlink,postdate,collected_at
0,3,아디다스,아디다스,naver_blog,"아디다스 아디제로 아디오스 프로4 사이즈 팁, 프로3, evo sl...",https://blog.naver.com/nudlelog/224241858954,"저는 주로 아디다스 러닝화를 많이 갖고있는데, 원래 신던 카본화 사이즈가 애매해서 ...",누들로그,blog.naver.com/nudlelog,2004-01-06,2026-04-27 10:00:34.953
1,3,아디다스,아디다스,naver_blog,중국 상해 여행 난징동루 쇼핑 보행자 거리 중티 아디다스 매장,https://blog.naver.com/tnwlsdl702/224249216536,쇼핑거리 아디다스 매장 원래는 난징동루 보행자 거리에서도 플래그쉽 매장으로 가려고 ...,책상에서 즐기는 여행 이야기,blog.naver.com/tnwlsdl702,2004-01-12,2026-04-27 10:00:34.953
2,3,아디다스,아디다스,naver_blog,아디다스 골프화 코드케이오스 필요 대상,https://blog.naver.com/sjrnfl81/224200392483,"&quot;이 포스팅은 네이버 쇼핑 커넥트 활동의 일환으로, 판매 발생 시 수수료를...",또 다른 세상,blog.naver.com/sjrnfl81,2003-01-01,2026-04-27 10:00:34.953
3,3,아디다스,아디다스,naver_blog,아디다스 도쿄 JI0183 사이즈 팁과 접지력 좋은 스니커즈...,https://manimo.tistory.com/348,🔗 함께 보면 좋은 상품 할인율 20% [아디다스 공식] 도쿄 JI0183 판매가 ...,manimo,https://manimo.tistory.com/,2004-01-08,2026-04-27 10:00:34.953
4,3,아디다스,아디다스,naver_blog,김해아울렛 나이키 아디다스 라코스테 세일 추가 할인 20~30프로,https://blog.naver.com/bbiccu6038/224256544357,"나이키, 아디다스, 라코스테 이렇게 세 군데만 집중해서 둘러봤어요. 먼저 아디다스 ...",맛집찾아삼만리,blog.naver.com/bbiccu6038,2004-01-18,2026-04-27 10:00:34.953
...,...,...,...,...,...,...,...,...,...,...,...
136,44,아틀리에 나인,아틀리에 나인,naver_blog,아틀리에 나인 24FALL 도산 플래그십 스토어 오픈 압구정...,https://blog.naver.com/eunryori_/223567738648,오늘은 얼마 전에 다녀왔던 아틀리에 나인 도산 플래그십 스토어 이야기 해드리려구 왔...,은료리 블로그,blog.naver.com/eunryori_,2008-01-31,2026-04-27 11:15:10.798
137,44,아틀리에 나인,아틀리에 나인,naver_blog,아틀리에 나인 부산 신세계센텀시티점 매장 오픈! 여자 가을 옷...,https://blog.naver.com/iamheozzang/223999445395,생각했던 아틀리에 나인도 들어왔어요 이렇게 백화점에서 매장에 가본 건 처음이에요 부...,디어진 블로그,blog.naver.com/iamheozzang,2009-01-08,2026-04-27 11:15:10.798
138,44,아틀리에 나인,아틀리에 나인,naver_blog,"기간 언제? 애플, 아틀리에나인, 노스페이스 혜택받자!",https://blog.naver.com/lqud/224064214961,"제가 기대하고 있는 브랜드가 또 있는데요, 11월 8일(토) 아틀리에나인! 특유의 ...",영업왕 제제의 블로그,blog.naver.com/lqud,2011-01-04,2026-04-27 11:15:10.798
139,44,아틀리에 나인,아틀리에 나인,naver_blog,차정원픽 아틀리에 나인 여자 부클 코트 추천,https://blog.naver.com/kuroinu92/224075118148,차정원픽 아틀리에 나인 부클 코트 아틀리에 나인이 코트나 니트 잘 만들기로 좀 유명...,‘마른체형’ 남자 코디 아카이브,blog.naver.com/kuroinu92,2011-01-13,2026-04-27 11:15:10.798
